# 第 15 节：A2C (Advantage Actor-Critic)

## 位置
Actor-Critic (14) → **A2C (15)** → PPO (16)...

## 学习目标
1. 理解从 1-step AC 到 n-step AC (A2C) 的演进
2. 掌握 A2C 架构：共享特征提取器、Actor 头、Critic 头
3. 理解 n-step return 与 MC return、TD(0) 的区别和联系
4. 理解 A2C vs A3C 的同步/异步区别
5. 从零实现完整的 A2C 算法
6. 理解熵正则化的作用
7. 在 CartPole 上训练并分析各组件的行为
8. 与 REINFORCE 和基础 AC 进行对比

## 1. 从 1-step AC 到 n-step AC (A2C)

### 1-step Actor-Critic (基础 AC)

1-step AC 使用 TD(0) 作为优势估计：

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

**问题**：TD(0) 的偏差大，信息只传播 1 步，学习速度慢。

### N-step Actor-Critic (A2C)

A2C 使用 n-step 回报作为优势估计，平衡偏差和方差：

$$R_t^{(n)} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n})$$

$$A_t^{(n)} = R_t^{(n)} - V(s_t)$$

### 偏差-方差权衡

| 方法 | 偏差 | 方差 | 信息传播速度 |
|------|------|------|-------------|
| TD(0) — 1-step | 高 | 低 | 慢 |
| MC — full episode | 无 | 高 | 快（但噪声大） |
| **N-step** | **可控** | **可控** | **折中** |

### 从基础 AC 到 A2C 的变化

1. **n-step rollout**：收集 n 步经验后统一更新
2. **优势计算**：使用 n-step return 替代 1-step TD error
3. **共享网络**：Actor 和 Critic 共享特征提取器
4. **熵正则化**：增加策略熵损失鼓励探索

## 2. A2C 架构

### 网络结构

```
输入状态 s (state_dim)
       |
  +----+----+
  |  Shared  |  共享特征提取器 (MLP)
  |  Layers  |
  +----+----+
       |
    +--+--+
    |     |
  Actor  Critic
    |     |
  logits  V(s)
  (n_actions)  (1)
```

### 共享特征提取器的好处

- 参数效率更高（共用特征）
- 特征表示同步学习
- 训练更稳定（Actor 和 Critic 互相正则化）

### 损失函数

$$\mathcal{L}_{A2C} = \mathcal{L}_{policy} + \mathcal{L}_{value} - \alpha \cdot \mathcal{H}(\pi)$$

其中：

- $\mathcal{L}_{policy} = -\frac{1}{N} \sum \log \pi(a_t|s_t) \cdot A_t$（策略损失）
- $\mathcal{L}_{value} = \frac{1}{N} \sum (R_t^{(n)} - V(s_t))^2$（价值损失）
- $\mathcal{H}(\pi) = -\frac{1}{N} \sum \sum \pi(a|s) \log \pi(a|s)$（熵奖励）
- $\alpha$：熵系数

## 3. N-step Return vs MC Return vs TD(0)

### 对比示意图

```
  t=0    t=1    t=2    ...    t=n-1    t=n     t=n+1   ...    t=T
  s0 --> s1 --> s2 --> ... --> s_{n-1} --> sn --> s_{n+1} --> ... --> sT
  a0     a1     a2             a_{n-1}   an
  r0     r1     r2             r_{n-1}   rn

  TD(0) 更新:  r0 + gamma V(s1)
  +-- 只用 1 步实际奖励 --+

  2-step 更新: r0 + gamma r1 + gamma^2 V(s2)
  +-- 只用 2 步实际奖励 ----+

  N-step 更新: r0 + gamma r1 + ... + gamma^{n-1}r_{n-1} + gamma^{n}V(sn)
  +-------- n 步实际奖励 --------+-- 引导值 --+

  MC 更新:    r0 + gamma r1 + ... + gamma^{T-1}r_{T-1}
  +-------- 整条 episode 的实际奖励 -------------+
```

### 关键公式

$$R_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$$

- 当 **n=1**：退化为 TD(0) — 高偏差，低方差
- 当 **n=infinity** (episode结束)：退化为 MC — 无偏，高方差
- **n 的选择**：典型的 A2C 使用 n=5 或 n=16

## 4. A2C vs A3C：同步与异步

### A3C (Asynchronous Advantage Actor-Critic)

- 多个 worker 独立与环境交互，**异步**更新全局网络
- 每个 worker 有自己的环境副本和网络副本
- 参数更新可能**过时** (stale gradients)
- 需要额外的同步机制

### A2C (Advantage Actor-Critic)

- 多个 worker 并行收集经验，**同步**更新
- 等所有 worker 完成 n-step 后再统一梯度更新
- 更新更稳定，梯度更一致
- GPU 利用率更高（可 batch 处理）

### 图示

```
A3C (异步):
Worker 1: -----A----->[Global]
Worker 2:   -----B------->[Network]
Worker 3:     -----C--------->[...]
         独立更新, 可能冲突

A2C (同步):
Worker 1: -----A-----|
Worker 2: -----B-----|----> [Gradient Aggregation] ---> [Update]
Worker 3: -----C-----|
         同步等待, 统一更新
```

在实践中，A2C 通常比 A3C 更稳定，训练效率更高，因此 A2C 成为更常用的变体。

In [ ]:
# Cell 6: 导入与配置

import sys; sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from rl_course.utils.seeding import set_seed; set_seed(42)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 配置 matplotlib
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 100})

FIG_DIR = "outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 环境
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f"CartPole: state_dim={state_dim}, n_actions={n_actions}")
env.close()


In [ ]:
# Cell 7: A2C 从零实现

from rl_course.networks.mlp import ActorCriticNetwork

class A2C:
    """Advantage Actor-Critic 从零实现

    特点:
    - 共享特征提取器的 Actor-Critic 网络
    - n-step 回报
    - 熵正则化
    """

    def __init__(
        self,
        state_dim: int,
        n_actions: int,
        hidden_dims: list = [128, 128],
        lr: float = 1e-3,
        gamma: float = 0.99,
        n_steps: int = 5,
        entropy_coef: float = 0.01,
    ):
        self.gamma = gamma
        self.n_steps = n_steps
        self.entropy_coef = entropy_coef

        # 使用 rl_course 提供的 ActorCriticNetwork
        # 返回 (action_logits, state_value)
        self.actor_critic = ActorCriticNetwork(
            state_dim=state_dim,
            n_actions=n_actions,
            hidden_dims=hidden_dims,
        ).to(device)

        self.optimizer = optim.Adam(self.actor_critic.parameters(), lr=lr)
        self.reset_buffer()

    def reset_buffer(self):
        """清空 rollout 缓存"""
        self.states = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.terminateds = []

    def act(self, state, train=True):
        """选择动作"""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        if train:
            with torch.no_grad():
                logits, _ = self.actor_critic(state_t)
                probs = F.softmax(logits, dim=-1)
                action = torch.multinomial(probs, 1).squeeze(-1).item()
            self.states.append(state_t.squeeze(0))
            self.actions.append(action)
            return action
        else:
            with torch.no_grad():
                logits, _ = self.actor_critic(state_t)
                action = torch.argmax(logits, dim=-1).item()
            return action

    def update(self, next_obs=None):
        """用 n-step 数据更新网络"""
        if len(self.states) == 0:
            return {"policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0}

        # 1. 转换为张量
        states = torch.stack(self.states).to(device)
        actions = torch.LongTensor(self.actions).to(device)
        rewards = torch.FloatTensor(self.rewards).to(device)
        dones = torch.FloatTensor(self.dones).to(device)
        terminateds = torch.FloatTensor(self.terminateds).to(device)

        # 2. 前向传播 (保留计算图)
        logits, values = self.actor_critic(states)
        values = values.squeeze(-1)

        # 3. 计算 log_prob 和熵
        log_probs_all = F.log_softmax(logits, dim=-1)
        probs = F.softmax(logits, dim=-1)
        action_log_probs = log_probs_all.gather(1, actions.unsqueeze(-1)).squeeze(-1)
        entropy = -(probs * log_probs_all).sum(dim=-1).mean()

        # 4. 计算 n-step 回报
        if next_obs is not None and not self.terminateds[-1]:
            next_state_t = torch.FloatTensor(next_obs).unsqueeze(0).to(device)
            with torch.no_grad():
                _, bootstrap_value = self.actor_critic(next_state_t)
            G = bootstrap_value.squeeze(-1).item()
        else:
            G = 0.0

        returns = []
        for i in reversed(range(len(rewards))):
            term = terminateds[i]
            G = rewards[i] + self.gamma * G * (1.0 - term)
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)

        # 5. 计算优势
        advantages = returns - values
        if advantages.numel() > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)

        # 6. 计算损失
        policy_loss = -(action_log_probs * advantages.detach()).mean()
        value_loss = F.mse_loss(values, returns)
        total_loss = policy_loss + value_loss - self.entropy_coef * entropy

        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor_critic.parameters(), 10.0)
        self.optimizer.step()

        info = {
            "policy_loss": policy_loss.item(),
            "value_loss": value_loss.item(),
            "entropy": entropy.item(),
        }
        self.reset_buffer()
        return info


In [ ]:
# Cell 8: N-step rollout 收集函数

def collect_n_step_rollout(env, agent, obs, n_steps=5):
    """收集 n 步经验的辅助函数

    每步调用 agent.act(train=True) 来记录状态和动作，
    然后在 step 后把 reward、done 和 terminated 存入 agent 的缓存。

    Args:
        obs: 当前观测（来自训练循环）

    Args:
        obs: 当前观测（来自训练循环）

    Returns:
        next_obs: 第 n 步后的观测（用于引导值计算）
        episode_ended: 是否在 rollout 期间 episode 结束
    """
    episode_ended = False
    if isinstance(obs, tuple):
        obs = obs[0]

    for _ in range(n_steps):
        action = agent.act(obs, train=True)
        result = env.step(action)

        if len(result) == 5:
            next_obs, reward, terminated, truncated, _ = result
            done = terminated or truncated
        else:
            next_obs, reward, done, _ = result

        agent.rewards.append(reward)
        agent.dones.append(done)
        agent.terminateds.append(terminated)

        if done:
            episode_ended = True
            if terminated:
                next_obs = None
            break

        obs = next_obs

    return next_obs, episode_ended


## 关于 A2C 的说明

**注意**：本 Notebook 实现的是**单环境 n-step Actor-Critic**，而非完整标准 A2C (Advantage Actor-Critic)。

标准 A2C 使用 `gym.vector.SyncVectorEnv` 并行收集多个环境的 n-step 经验，然后统一计算梯度更新。本实现为了教学简洁性，仅使用单个环境串行采集 n 步。

主要区别：
- **标准 A2C**：多个 worker 同步收集，梯度汇总更新
- **本实现**：单环境收集 n 步后更新，适合教学演示

以下训练代码调用 `collect_n_step_rollout(env, agent, obs, n_steps=N_STEPS)`，其中 `obs` 来自外层循环维护的当前观测。

In [ ]:
# Cell 9: A2C 训练循环

N_STEPS = 8
GAMMA = 0.99
LR = 3e-4
ENTROPY_COEF = 0.01

agent = A2C(
    state_dim=state_dim,
    n_actions=n_actions,
    hidden_dims=[128, 128],
    lr=LR,
    gamma=GAMMA,
    n_steps=N_STEPS,
    entropy_coef=ENTROPY_COEF,
)

metrics = {
    "episode_returns": [],
    "policy_losses": [],
    "value_losses": [],
    "entropies": [],
    "episode_lengths": [],
}

set_seed(42)

print(f"训练 A2C (n_steps={N_STEPS}, entropy_coef={ENTROPY_COEF})...")
episode_return = 0.0
episode_length = 0

env = gym.make("CartPole-v1")
obs, _ = env.reset()

for step in tqdm(range(20000)):
    next_obs, episode_ended = collect_n_step_rollout(env, agent, obs, n_steps=N_STEPS)

    for r in agent.rewards:
        episode_return += r
        episode_length += 1

    info = agent.update(next_obs)

    metrics["policy_losses"].append(info["policy_loss"])
    metrics["value_losses"].append(info["value_loss"])
    metrics["entropies"].append(info["entropy"])

    if episode_ended:
        metrics["episode_returns"].append(episode_return)
        metrics["episode_lengths"].append(episode_length)
        episode_return = 0.0
        episode_length = 0
        obs, _ = env.reset()
    else:
        obs = next_obs

env.close()

print(f"\n训练完成！共 {len(metrics['episode_returns'])} 个 episode")
if len(metrics["episode_returns"]) > 0:
    print(f"平均 return: {np.mean(metrics['episode_returns'][-50:]):.2f}")


In [ ]:
# Cell 10: A2C 训练曲线

%matplotlib inline

def smooth(data, window=10):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

window = 5

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# 1. Episode Returns
axes[0, 0].plot(metrics["episode_returns"], alpha=0.3, linewidth=0.5, color='steelblue')
if len(metrics["episode_returns"]) > window:
    axes[0, 0].plot(range(window-1, len(metrics["episode_returns"])),
                    smooth(metrics["episode_returns"], window),
                    linewidth=2, color='steelblue')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Return')
axes[0, 0].set_title('Episode Returns')
axes[0, 0].grid(True, alpha=0.3)

# 2. Policy Loss
axes[0, 1].plot(metrics["policy_losses"], alpha=0.3, linewidth=0.5, color='coral')
if len(metrics["policy_losses"]) > window:
    axes[0, 1].plot(range(window-1, len(metrics["policy_losses"])),
                    smooth(metrics["policy_losses"], window),
                    linewidth=2, color='coral')
axes[0, 1].set_xlabel('Update Step')
axes[0, 1].set_ylabel('Policy Loss')
axes[0, 1].set_title('Actor Loss')
axes[0, 1].grid(True, alpha=0.3)

# 3. Value Loss
axes[0, 2].plot(metrics["value_losses"], alpha=0.3, linewidth=0.5, color='green')
if len(metrics["value_losses"]) > window:
    axes[0, 2].plot(range(window-1, len(metrics["value_losses"])),
                    smooth(metrics["value_losses"], window),
                    linewidth=2, color='green')
axes[0, 2].set_xlabel('Update Step')
axes[0, 2].set_ylabel('Value Loss (MSE)')
axes[0, 2].set_title('Critic Loss')
axes[0, 2].grid(True, alpha=0.3)

# 4. Entropy
axes[1, 0].plot(metrics["entropies"], alpha=0.3, linewidth=0.5, color='purple')
if len(metrics["entropies"]) > window:
    axes[1, 0].plot(range(window-1, len(metrics["entropies"])),
                    smooth(metrics["entropies"], window),
                    linewidth=2, color='purple')
axes[1, 0].set_xlabel('Update Step')
axes[1, 0].set_ylabel('Entropy')
axes[1, 0].set_title('Policy Entropy (鼓励探索)')
axes[1, 0].grid(True, alpha=0.3)

# 5. Episode Lengths
axes[1, 1].plot(metrics["episode_lengths"], alpha=0.3, linewidth=0.5, color='orange')
if len(metrics["episode_lengths"]) > window:
    axes[1, 1].plot(range(window-1, len(metrics["episode_lengths"])),
                    smooth(metrics["episode_lengths"], window),
                    linewidth=2, color='orange')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Length')
axes[1, 1].set_title('Episode Length')
axes[1, 1].grid(True, alpha=0.3)

# 6. Return histogram
axes[1, 2].hist(metrics["episode_returns"], bins=20, alpha=0.7, color='steelblue', edgecolor='white')
axes[1, 2].set_xlabel('Return')
axes[1, 2].set_ylabel('Count')
axes[1, 2].set_title('Return Distribution')
axes[1, 2].grid(True, alpha=0.3, axis='y')

fig.suptitle('A2C Training on CartPole-v1', fontsize=14)
fig.tight_layout()
filepath = os.path.join(FIG_DIR, "15_a2c_training_curves.png")
fig.savefig(filepath)
plt.close(fig)
print(f"Figure saved: {filepath}")


In [ ]:
# Cell 11: 算法对比 — REINFORCE vs Actor-Critic vs A2C

from rl_course.agents.reinforce import REINFORCEAgent
from rl_course.agents.a2c import A2CAgent

N_TRAIN_EPISODES = 200

def evaluate(agent, env_name="CartPole-v1", n_episodes=20):
    env = gym.make(env_name)
    returns = []
    for _ in range(n_episodes):
        state, _ = env.reset()
        episode_return = 0.0
        done = False
        while not done:
            action = agent.act(state, train=False)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_return += reward
            done = terminated or truncated
            state = next_state
        returns.append(episode_return)
    env.close()
    return np.mean(returns), np.std(returns)

set_seed(42)

# 1. REINFORCE
print("训练 REINFORCE...")
reinforce_agent = REINFORCEAgent(state_dim, n_actions, gamma=0.99, lr=3e-4)
for ep in tqdm(range(N_TRAIN_EPISODES), desc="REINFORCE"):
    env = gym.make("CartPole-v1")
    state, _ = env.reset()
    done = False
    while not done:
        action = reinforce_agent.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        reinforce_agent.episode_rewards.append(reward)
        done = terminated or truncated
        state = next_state
    reinforce_agent.update()
    env.close()
mean_reinf, std_reinf = evaluate(reinforce_agent)
print(f"REINFORCE: {mean_reinf:.2f} +/- {std_reinf:.2f}")

# 2. Actor-Critic (1-step)
print("\n训练 1-step Actor-Critic...")
ac_agent = A2CAgent(state_dim, n_actions, gamma=0.99, lr=3e-4, n_steps=1)
for ep in tqdm(range(N_TRAIN_EPISODES), desc="AC (1-step)"):
    env = gym.make("CartPole-v1")
    state, _ = env.reset()
    done = False
    while not done:
        action = ac_agent.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        ac_agent.rewards.append(reward)
        ac_agent.dones.append(terminated or truncated)
        done = terminated or truncated
        if done:
            ac_agent.update(next_obs=None)
        else:
            state = next_state
    if not done:
        ac_agent.update(next_obs=next_state)
    env.close()
mean_ac, std_ac = evaluate(ac_agent)
print(f"Actor-Critic (1-step): {mean_ac:.2f} +/- {std_ac:.2f}")

# 3. A2C (n-step)
mean_a2c, std_a2c = evaluate(agent)
print(f"A2C (n-step): {mean_a2c:.2f} +/- {std_a2c:.2f}")

# 可视化对比
%matplotlib inline
fig, ax = plt.subplots(figsize=(10, 5))
algorithms = ['REINFORCE (MC)', 'Actor-Critic (1-step)', 'A2C (n-step)']
means = [mean_reinf, mean_ac, mean_a2c]
stds = [std_reinf, std_ac, std_a2c]
colors = ['steelblue', 'coral', 'seagreen']

bars = ax.bar(algorithms, means, yerr=stds, color=colors, alpha=0.8, capsize=8, width=0.5)
ax.axhline(y=500, color='gray', linestyle='--', alpha=0.7, label='Max Return (500)')
ax.set_ylabel(f'Average Return (over 20 episodes)')
ax.set_title('Algorithm Comparison on CartPole-v1')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{mean:.1f}\n+/-{std:.1f}', ha='center', fontsize=11)

filepath = os.path.join(FIG_DIR, "15_algorithm_comparison.png")
fig.savefig(filepath)
plt.close(fig)
print(f"\nFigure saved: {filepath}")


## 5. 对比总结：REINFORCE vs AC vs A2C

| 特性 | REINFORCE | 1-step AC | A2C (n-step) |
|------|-----------|-----------|--------------|
| **更新信号** | MC return $G_t$ | TD(0) $\delta_t$ | n-step return $R_t^{(n)}$ |
| **偏差** | 无 | 高 | 可控（n 越大偏差越小） |
| **方差** | 高 | 低 | 可控（n 越大方差越大） |
| **更新时机** | episode 结束 | 每一步 | 每 n 步 |
| **熵正则化** | 通常无 | 可以有 | 标准组件 |
| **学习速度** | 慢（等整条 episode） | 快（在线更新） | 中等 |
| **稳定性** | 差 | 中等 | 好 |
| **实现复杂度** | 低 | 中 | 中高 |

### 启示

- **REINFORCE**：无偏但方差大，适合简单环境
- **1-step AC**：方差小但偏差大，可能学到错误的价值估计
- **A2C**：通过 n 参数灵活控制偏差-方差平衡，是实际中最常用的变体

In [ ]:
# Cell 13: 录制 A2C 智能体运行视频

from rl_course.visualization.video import record_episode

env_render = gym.make("CartPole-v1", render_mode="rgb_array")

def policy_fn(state):
    return agent.act(state, train=False)

video_path = record_episode(
    env=env_render,
    policy_fn=policy_fn,
    fps=30,
    max_steps=500,
    format="gif",
    verbose=True,
)

env_render.close()
print(f"\n视频已录制: {video_path}")


## 6. A2C 实现常见陷阱

### 陷阱 1：忘记 detach 优势

```python
# 错误！策略梯度会错误地更新价值网络
policy_loss = -(log_prob * advantages).mean()

# 正确！detach 切断计算图
policy_loss = -(log_prob * advantages.detach()).mean()
```

### 陷阱 2：优势归一化时机

```python
# 推荐：在 detach 之前归一化，数值更稳定
advantages = (returns - values)
advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
policy_loss = -(log_prob * advantages.detach()).mean()
```

### 陷阱 3：价值网络引导值处理

```python
# 如果 episode 在 n-step 内结束，引导值必须为 0
if done:
    bootstrap_value = 0.0  # episode 结束，没有未来回报
else:
    bootstrap_value = V(s_{t+n})  # 用价值网络估计
```

### 陷阱 4：熵系数过小或过大

- 过小 (0.0001)：策略过早确定，可能陷入次优
- 过大 (0.1)：策略始终保持高熵（随机），难以收敛
- 推荐：0.01 作为起点，观察熵曲线调整

### 陷阱 5：n 的选择

- n 太小 (1)：偏差大，学习慢
- n 太大 (32+)：方差大，不稳定
- 推荐：CartPole 用 5-16，Atari 用 5

### 陷阱 6：共享网络 vs 独立网络

共享特征提取器更高效，但 Actor 和 Critic 的梯度相互影响。有时使用独立网络更稳定但更慢。

In [ ]:
# Cell 15: 可视化 detach 的重要性

%matplotlib inline

from rl_course.networks.mlp import ActorCriticNetwork

d_state_dim = 4
d_n_actions = 2

net_bad = ActorCriticNetwork(d_state_dim, d_n_actions, [64]).to(device)
net_good = ActorCriticNetwork(d_state_dim, d_n_actions, [64]).to(device)
net_good.load_state_dict(net_bad.state_dict())

opt_bad = optim.Adam(net_bad.parameters(), lr=1e-3)
opt_good = optim.Adam(net_good.parameters(), lr=1e-3)

value_losses_bad = []
value_losses_good = []

set_seed(42)
dummy_states = torch.randn(8, d_state_dim).to(device)
dummy_actions = torch.randint(0, d_n_actions, (8,)).to(device)
dummy_returns = torch.randn(8).to(device) * 2 + 1

for step in range(200):
    # Bad: no detach
    logits_b, values_b = net_bad(dummy_states)
    values_b = values_b.squeeze(-1)
    advantages_b = dummy_returns - values_b
    log_probs_b = F.log_softmax(logits_b, dim=-1)
    action_log_probs_b = log_probs_b.gather(1, dummy_actions.unsqueeze(-1)).squeeze(-1)
    policy_loss_b = -(action_log_probs_b * advantages_b).mean()  # no detach!
    value_loss_b = F.mse_loss(values_b, dummy_returns)
    total_loss_b = policy_loss_b + value_loss_b
    opt_bad.zero_grad()
    total_loss_b.backward()
    opt_bad.step()

    # Good: with detach
    logits_g, values_g = net_good(dummy_states)
    values_g = values_g.squeeze(-1)
    advantages_g = dummy_returns - values_g
    log_probs_g = F.log_softmax(logits_g, dim=-1)
    action_log_probs_g = log_probs_g.gather(1, dummy_actions.unsqueeze(-1)).squeeze(-1)
    policy_loss_g = -(action_log_probs_g * advantages_g.detach()).mean()  # detach!
    value_loss_g = F.mse_loss(values_g, dummy_returns)
    total_loss_g = policy_loss_g + value_loss_g
    opt_good.zero_grad()
    total_loss_g.backward()
    opt_good.step()

    value_losses_bad.append(value_loss_b.item())
    value_losses_good.append(value_loss_g.item())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(value_losses_bad, alpha=0.7, label='No Detach (震荡)', color='red')
ax.plot(value_losses_good, alpha=0.7, label='With Detach (收敛)', color='green')
ax.set_xlabel('Update Step')
ax.set_ylabel('Value Loss (MSE)')
ax.set_title('Detach 的重要性：Value Loss 对比')
ax.legend()
ax.grid(True, alpha=0.3)

filepath = os.path.join(FIG_DIR, "15_detach_importance.png")
fig.savefig(filepath)
plt.close(fig)
print(f"Figure saved: {filepath}")
print("结论：不 detach 导致 value loss 无法收敛，训练发散！")


In [ ]:
# Cell 16: 不同 n-step 步数的效果对比

%matplotlib inline

set_seed(42)

def train_a2c_quick(n_steps, n_episodes=100):
    agent_q = A2C(state_dim, n_actions, hidden_dims=[64, 64],
                  lr=3e-4, gamma=0.99, n_steps=n_steps, entropy_coef=0.01)
    env = gym.make("CartPole-v1")
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset()
        episode_return = 0.0
        done = False
        steps = 0
        while not done and steps < 500:
            action = agent_q.act(obs, train=True)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent_q.rewards.append(reward)
            agent_q.dones.append(done)
            agent_q.terminateds.append(terminated)
            steps += 1
            episode_return += reward
            if len(agent_q.states) >= n_steps or done:
                nxt = next_obs if not done else None
                agent_q.update(nxt)
            obs = next_obs
        returns.append(episode_return)
    env.close()
    return np.mean(returns[-50:]) if len(returns) >= 50 else np.mean(returns)

n_values = [1, 4, 8, 16, 32]
mean_returns = []

print("对比不同 n-step 大小（训练 100 episodes）...")
for n in n_values:
    mean_ret = train_a2c_quick(n_steps=n, n_episodes=100)
    mean_returns.append(mean_ret)
    print(f"  n={n:2d}: Avg Return = {mean_ret:.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_values, mean_returns, marker='o', linewidth=2, color='steelblue', markersize=8)
ax.set_xlabel('n-step size')
ax.set_ylabel('Average Return')
ax.set_title('Effect of n-step Size on A2C Performance')
ax.grid(True, alpha=0.3)
ax.set_xticks(n_values)

best_idx = int(np.argmax(mean_returns))
ax.annotate(f'Best: n={n_values[best_idx]}', xy=(n_values[best_idx], mean_returns[best_idx]),
            xytext=(n_values[best_idx] + 5, mean_returns[best_idx] - 20),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')

filepath = os.path.join(FIG_DIR, "15_nstep_comparison.png")
fig.savefig(filepath)
plt.close(fig)
print(f"\nFigure saved: {filepath}")


## 7. 总结

### 核心要点

1. **A2C 的核心思想**：使用 n-step 回报 $R_t^{(n)}$ 作为优势估计，在偏差和方差之间取得平衡

2. **架构特点**：
   - 共享特征提取器的 Actor-Critic 网络
   - n-step rollout 收集经验
   - 熵正则化鼓励探索

3. **A2C 的关键公式**：

   | 组件 | 公式 |
   |------|------|
   | n-step return | $R_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$ |
   | 优势函数 | $A_t = R_t^{(n)} - V(s_t)$ |
   | 策略损失 | $\mathcal{L}_{policy} = -\log \pi(a_t|s_t) \cdot A_t$ |
   | 价值损失 | $\mathcal{L}_{value} = (R_t^{(n)} - V(s_t))^2$ |
   | 总损失 | $\mathcal{L} = \mathcal{L}_{policy} + \mathcal{L}_{value} - \alpha \mathcal{H}(\pi)$ |

4. **A2C vs A3C**：
   - A2C：同步，稳定，GPU 友好
   - A3C：异步，更快收集经验，但梯度可能过时

5. **实现关键点**：
   - 必须 detach 优势再计算策略损失
   - 正确处理 episode 结束时的引导值
   - 熵正则化防止过早收敛

### 学习路径

```
REINFORCE -> REINFORCE + Baseline -> Actor-Critic (1-step) -> A2C (n-step) -> PPO (clipped)
     (12)        (13)                   (14)                  (15)          (16)
```

## 8. 练习

### 基础练习

1. **n-step 的影响**：尝试 n=1, 2, 4, 8, 16, 32，绘图对比最终性能和收敛速度
2. **熵系数的影响**：尝试 entropy_coef=0, 0.001, 0.01, 0.1，观察策略熵和最终性能
3. **学习率的影响**：尝试不同的学习率，找到 CartPole 上最适合的学习率

### 进阶练习

4. **实现 GAE**：将 n-step return 替换为 GAE，比较效果
   $$A_t^{GAE} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$
5. **独立网络 vs 共享网络**：实现不共享特征的版本，比较参数效率和学习效果
6. **优势归一化**：去掉 advantages 归一化，观察训练是否还能稳定收敛

### 扩展练习

7. **更复杂的环境**：在 LunarLander-v2 或 Acrobot-v1 上运行 A2C
8. **A2C vs A3C 实现**：用 Python 的 multiprocessing 实现简单的 A3C，比较同步和异步的差异
9. **奖励缩放**：实现奖励裁剪或奖励归一化，观察效果

### 思考题

10. 为什么 A2C 中 Actor 和 Critic 共享特征提取器有时会导致训练不稳定？
11. 如果 n-step 的 n 非常大（接近 episode 长度），A2C 退化成什么？
12. 熵正则化的系数应该如何动态调整？